In [27]:
import numpy as np
import matplotlib.pyplot as plt
import datasets

In [28]:
mnist_ds = datasets.load_from_disk("ylecun/mnist").with_format("numpy")

# 2. Extract contiguous blocks directly (instant PyArrow extraction)
X_train = (mnist_ds["train"]["image"][:] / 255.0).astype(np.float32)
Y_train = mnist_ds["train"]["label"][:].astype(np.int64)

X_test = (mnist_ds["test"]["image"][:] / 255.0).astype(np.float32)
Y_test = mnist_ds["test"]["label"][:].astype(np.int64)

# 3. Fast in-memory shuffle using NumPy
rng = np.random.default_rng(42)

train_idx = rng.permutation(len(X_train))
X_train, Y_train = X_train[train_idx], Y_train[train_idx]

test_idx = rng.permutation(len(X_test))
X_test, Y_test = X_test[test_idx], Y_test[test_idx]

# one hotting
Y_train = np.eye(10, dtype=np.float32)[Y_train]   
Y_test = np.eye(10, dtype=np.float32)[Y_test]     

# flatening and changing resolution
X_train = X_train.reshape(-1, 14, 2, 14, 2).mean(axis=(2, 4)).reshape(len(X_train), -1)  
X_test = X_test.reshape(-1, 14, 2, 14, 2).mean(axis=(2, 4)).reshape(len(X_test), -1)     

In [ ]:
from abc import ABC, abstractmethod
from typing import Iterator
from numpy.typing import NDArray

class Trainer():
    def __init__(self, optimizer: Optimizer, x_train_set=NDArray, y_train_set=NDArray , batching=False, chunk_size=-1, batch_size = -1):
        self.optimizer = optimizer
        self.batching = batching
        self.chunk_size = chunk_size
        self.batch_size = batch_size
        self.x_train_set = x_train_set
        self.y_train_set = y_train_set

    def train(self, n_steps=-1, report_freq=1): # each step is composed of n chunks or batch elements
        for i in range(n_steps):
            loss = 0
            for j in range(self.chunk_size):
                loss += self.optimizer.step_accumulate_gradients(self.x_train_set[i * self.chunk_size + j], self.y_train_set[i * self.chunk_size + j])
            self.optimizer.step_learn()

            loss /= self.chunk_size
            if i % report_freq == 0: print(f"step: {i} loss = {loss}")



class Model(ABC):

    layers: list

    @abstractmethod
    def accumulate_gradients(self, gradients_forward): pass
    
    def zero_grad(self):
        for layer in self.layers: layer.zero_grad()

    @abstractmethod
    def __call__(self, x): pass

    @abstractmethod
    def parameters(self) -> Iterator[tuple[NDArray[np.float32], NDArray[np.float32]]]: pass

    
class Optimizer(ABC):
    
    @abstractmethod
    def step_accumulate_gradients(self, x, y) -> float: pass

    @abstractmethod
    def step_learn(self): pass

   


class Layer():
    def __init__(self, nin, nout, nonlin, seed=42): # eah nout is a neuron
        self._tmp = np.zeros((nout, nin), dtype=np.float32) # used as a buffer to avoid reallocating constantly
        self.nout = nout
        self.nin = nin
        rng = np.random.default_rng(seed)

        self.weights = rng.standard_normal((nout, nin), dtype=np.float32) * np.float32(np.sqrt(2.0 / nin)) 
        self.gradients = np.zeros((nout, nin), dtype=np.float32)

        self.bias_weights = rng.standard_normal((nout,), dtype=np.float32) * np.float32(np.sqrt(2.0 / nin)) 
        self.bias_gradients = np.zeros((nout,), dtype=np.float32)

        self.mask = np.ones((nout,), dtype=np.float32)
        self.nonlin = nonlin
        self.activationsin = np.zeros((nin,), dtype=np.float32)
        self.activationsout = np.zeros((nout,), dtype=np.float32)
        

    def __call__(self, activation):
        self.activationsin = activation
        neurons_out = (self.weights @ activation) + self.bias_weights
        if self.nonlin:
            self.mask = neurons_out > 0 # ReLu activation function
        self.activationsout = neurons_out * self.mask
        return self.activationsout
    
    def accumulate_gradients(self, gradients_forward):
        # gradients_forward is of shape [nout]
        # self.gradients is of shape [nout, nin]
        # self.activationsin of shape [nin] -> [nout, nin]

        gradients_forward = gradients_forward * self.mask # kills gradients of dead neurons

        # multiplication of 2 1d arrays, gradients forward * activations in of shape (nout, nin)

        np.outer(gradients_forward, self.activationsin, out=self._tmp)
        self.gradients += self._tmp

        # bias gradient have a local derivative thats always 1 so we only chain to upstream derivative
        self.bias_gradients += gradients_forward

        # gradient at each input, equals the sum of (each weight connected to it) multiplied by its upstream gradient (forward gradient)
        
        gradients_backward = gradients_forward @ self.weights
        return gradients_backward
        

    def zero_grad(self):
        self.gradients.fill(0)
        self.bias_gradients.fill(0)

    # def update_weights(self, learning_rate):
    #     np.multiply(learning_rate, self.gradients, out=self._tmp) # weights
        
    #     self.weights -= self._tmp
    #     self.bias_weights -= self.bias_gradients * learning_rate

    def parameters(self):
        yield self.weights, self.gradients
        yield self.bias_weights, self.bias_gradients

    def __repr__(self):
        nin = self.weights.shape[1]
        nout = self.weights.shape[0]
        
        lines = [f"Layer(nin={nin}, nout={nout}, nonlin={self.nonlin}):"]
        
        # Format weights matrix and bias column
        for i in range(nout):
            # Format each weight with fixed total width (e.g., 7 chars, 3 decimals, space for sign)
            w_row = " ".join(f"{w: 7.3f}" for w in self.weights[i])
            bias = f"{self.bias_weights[i]: 7.3f}"
            mask = f"{int(self.mask[i])}"
            
            lines.append(f"  Neuron {i}: W = [{w_row} ]  b = {bias}  mask = {mask}")
            
        return "\n".join(lines)

    
class MLP(Model):
    def __init__(self, nin, nouts): # int, int[]
        self.layers = []

        # making all layers nonlin (relu) except the last

        if len(nouts) > 1: self.layers.append(Layer(nin=nin, nout=nouts[0], nonlin=True))
        else: self.layers.append(Layer(nin=nin, nout=nouts[0], nonlin=False))

        for i, width in enumerate(nouts[1:-1], start=1):
            self.layers.append(Layer(nin=nouts[i-1], nout=width, nonlin=True, seed=42 + i))

        if len(nouts) > 1: self.layers.append(Layer(nin=nouts[-2], nout=nouts[-1], nonlin=False))

    def __call__(self, activations):
        for layer in self.layers:
            activations = layer(activations)
        return activations

    def parameters(self):
        for layer in self.layers:
            yield from layer.parameters()
            

    def accumulate_gradients(self, gradients_forward):

        for layer in reversed(self.layers):
            gradients_forward = layer.accumulate_gradients(gradients_forward=gradients_forward)

    def __repr__(self):
            out = ""
            for k, layer in enumerate(self.layers):
                out += f"Layer {k}: {str(layer)}\n\n"
            return out
    
    def stats(self):
        header = f"{'Layer':<7}{'Inputs':>10}{'Neurons':>10}{'Params':>12}  {'Activation':<10}"
        line = "-" * len(header)

        print(line)
        print(header)
        print(line)

        total = 0
        for k, layer in enumerate(self.layers):
            params = layer.nout * (layer.nin + 1)   # weights + biases
            total += params
            act = "ReLU" if layer.nonlin else "Linear"
            print(f"{k:<7}{layer.nin:>10,}{layer.nout:>10,}{params:>12,}  {act:<10}")

        print(line)
        print(f"Layers: {len(self.layers)}")
        print(f"Total parameters: {total:,}")
        print(f"Memory (weights, float32): {total * 4 / 1e6:.2f} MB")
        print(line)


class SGD(Optimizer): # Standard Gradient Decent, dumb decent
    def __init__(self, model: Model, learning_rate):
        self.model = model
        self.learning_rate = learning_rate

    def step_accumulate_gradients(self, x, y): # accumulates allowing for chunking

        z = self.model(x)
        loss, grad = z_loss_and_gradient(z=z, y=y)
        self.model.accumulate_gradients(gradients_forward=grad)

        return loss
    
    def step_learn(self):
        for p, g in self.model.parameters():
            g *= self.learning_rate              # in place, no temp
            p -= g   
        self.model.zero_grad()





def z_loss_and_gradient(z, y):
    z_shifted = z - z.max()
    exp_z = np.exp(z_shifted)
    sum_exp = exp_z.sum()
    grad = exp_z / sum_exp - y
    loss = np.log(sum_exp) * y.sum() - y @ z_shifted   # -sum(y * log_softmax)
    return float(loss), grad

def z_loss(z, y): # only use when gradient is not needed, else prefer z_loss_and_gradient
    z_shifted = z - np.max(z)                      # for numerical stability
    log_probs = z_shifted - np.log(np.sum(np.exp(z_shifted)))   # log-softmax
    return -np.sum(y * log_probs)
    

In [34]:
correct = 0
test_loss = 0.0
for x, y in zip(X_test, Y_test):
    z = model(x)
    correct += z.argmax() == y.argmax()
    test_loss += z_loss(z, y)

print(f"test accuracy: {correct / len(X_test):.4f}  test loss: {test_loss / len(X_test):.4f}")

test accuracy: 0.9690  test loss: 0.0991


In [35]:
model = MLP(nin=14*14, nouts=[11*11, 8*8, 5*5, 10], )

optimizer = SGD(model=model, learning_rate=0.0005)

trainer = Trainer(optimizer=optimizer, x_train_set=X_train, y_train_set=Y_train, chunk_size=12)

%prun -s cumtime trainer.train(n_steps=5_000, report_freq=200)

step: 0 loss = 2.3601157665252686
step: 200 loss = 2.0967423915863037
step: 400 loss = 1.6216353178024292
step: 600 loss = 1.2174960374832153
step: 800 loss = 0.9954172968864441
step: 1000 loss = 1.0165990591049194
step: 1200 loss = 0.7873075008392334
step: 1400 loss = 0.8926247954368591
step: 1600 loss = 0.30179738998413086
step: 1800 loss = 0.24736028909683228
step: 2000 loss = 0.5121524930000305
step: 2200 loss = 0.88124680519104
step: 2400 loss = 0.22737263143062592
step: 2600 loss = 0.8790552616119385
step: 2800 loss = 0.541709840297699
step: 3000 loss = 0.7798089385032654
step: 3200 loss = 0.36302003264427185
step: 3400 loss = 0.15239354968070984
step: 3600 loss = 0.22232820093631744
step: 3800 loss = 0.48349082469940186
step: 4000 loss = 0.4327028691768646
step: 4200 loss = 0.17496727406978607
step: 4400 loss = 0.1880311220884323
step: 4600 loss = 0.14716172218322754
step: 4800 loss = 0.39276647567749023
 

         4148547 function calls (4148543 primitive calls) in 7.404 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
       78    0.078    0.001   14.546    0.186 base_events.py:1976(_run_once)
    60000    0.137    0.000    7.035    0.000 1707987795.py:196(step_accumulate_gradients)
      2/1    0.000    0.000    5.124    5.124 {built-in method builtins.exec}
        1    0.002    0.002    4.541    4.541 <string>:1(<module>)
    60000    0.163    0.000    3.568    0.000 1707987795.py:158(accumulate_gradients)
   240000    1.624    0.000    3.409    0.000 1707987795.py:81(accumulate_gradients)
       75    0.001    0.000    2.320    0.031 events.py:92(_run)
   240000    1.522    0.000    1.750    0.000 numeric.py:905(outer)
    60000    0.160    0.000    1.436    0.000 1707987795.py:148(__call__)
   240000    1.277    0.000    1.277    0.000 1707987795.py:73(__call__)
    60000    0.424    0.000    1.089    0.000 1707987795

In [32]:
optimizer.learning_rate= 0.0005
trainer.train(n_steps=5_000, report_freq=200)

step: 0 loss = 0.063254214823246
step: 200 loss = 0.08390715718269348
step: 400 loss = 0.06602659076452255
step: 600 loss = 0.012603439390659332
step: 800 loss = 0.07451155036687851
step: 1000 loss = 0.4352222681045532
step: 1200 loss = 0.12644408643245697
step: 1400 loss = 0.10192421078681946
step: 1600 loss = 0.007427473086863756
step: 1800 loss = 0.0030153775587677956
step: 2000 loss = 0.0900016650557518
step: 2200 loss = 0.09940425306558609
step: 2400 loss = 0.023916825652122498
step: 2600 loss = 0.04664064943790436
step: 2800 loss = 0.24869795143604279
step: 3000 loss = 0.10895033925771713
step: 3200 loss = 0.06457725912332535
step: 3400 loss = 0.010744132101535797
step: 3600 loss = 0.05717170611023903
step: 3800 loss = 0.06434169411659241
step: 4000 loss = 0.19518864154815674
step: 4200 loss = 0.008961268700659275
step: 4400 loss = 0.09460218995809555
step: 4600 loss = 0.04440000653266907
step: 4800 loss = 0.05186689272522926
